# V-JEPA 2 — Video Joint-Embedding Predictive Architecture

**V-JEPA** extends the JEPA idea from images to **video**: it predicts the
latent representations of masked spatio-temporal regions from visible ones,
learning motion and temporal structure without pixel reconstruction or labels.
**V-JEPA 2** (Meta AI, 2025) scales this up on internet-scale video and adds a
predictor head, making it strong for action understanding and as a world model.

### What this notebook covers
1. **Feature extraction** — load `AutoModel` and read both the encoder and
   predictor outputs for a video clip.
2. **Video inference (action recognition)** — use
   `AutoModelForVideoClassification` on a fine-tuned Something-Something-v2 head.
3. **Video classification** — the `VJEPA2ForVideoClassification` class directly,
   including how to compute a training loss.


### References
- HF model docs: https://huggingface.co/docs/transformers/main/model_doc/vjepa2
- Code: https://github.com/facebookresearch/vjepa2


### Feature extraction

https://huggingface.co/docs/transformers/main/model_doc/vjepa2 

In [1]:
# Confirm a GPU is present before loading the (large) video model.
!nvidia-smi

Sun Jul  5 15:51:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.71.05              Driver Version: 595.71.05      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A10G                    On  |   00000000:00:1E.0 Off |                    0 |
|  0%   31C    P0             60W /  300W |       0MiB /  23028MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Disable torchaudio's torchcodec path so video decoding uses torchcodec
# directly and avoids a backend clash.
import os 
# TORCHAUDIO_USE_TORCHCODEC=0
os.environ["TORCHAUDIO_USE_TORCHCODEC"] = "0"

In [3]:
# VideoDecoder streams frames straight from a URL or file path.
import numpy as np
from torchcodec.decoders import VideoDecoder

In [4]:
from transformers import AutoModel

In [5]:
from transformers import AutoVideoProcessor, VJEPA2ForVideoClassification, AutoModelForVideoClassification

The chosen checkpoint `vjepa2-vitl-fpc64-256` is a ViT-L encoder that consumes 64 frames at 256px. `AutoModel` loads the **backbone only** (encoder + predictor), which is what we want for raw feature extraction.

In [6]:
model_id = "facebook/vjepa2-vitl-fpc64-256"

In [7]:
# device_map="auto" shards the model onto available GPU(s); sdpa = fast attention.
processor = AutoVideoProcessor.from_pretrained(model_id)

model = AutoModel.from_pretrained(
    model_id,
    device_map="auto",
    attn_implementation="sdpa"
)

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

In [8]:
video_url = "https://huggingface.co/datasets/nateraw/kinetics-mini/resolve/main/val/archery/-Qz25rXdMjE_000014_000024.mp4"

In [9]:
# Grab the first 64 frames (T x C x H x W). Swap in a smarter sampler
# (e.g. uniform over the whole clip) for real use.
vr = VideoDecoder(video_url)
frame_idx = np.arange(0, 64) # choosing some frames. here, you can define more complex sampling strategy
video = vr.get_frames_at(indices=frame_idx).data  # T x C x H x W
video = processor(video, return_tensors="pt").to(model.device)
outputs = model(**video)

In [10]:
# Encoder features: per-patch spatio-temporal representations.
# V-JEPA 2 encoder outputs, same as calling `model.get_vision_features()`
encoder_outputs = outputs.last_hidden_state

In [11]:
encoder_outputs

tensor([[[-1.2721,  2.0601,  0.0382,  ..., -3.4707,  1.8614, -1.3039],
         [-1.1360, -0.6614, -0.7239,  ...,  0.8813,  0.0443, -0.8219],
         [ 0.1813,  1.3271, -1.1693,  ..., -0.1270, -0.5997, -1.3078],
         ...,
         [-0.7338,  2.4156,  1.1471,  ...,  2.9056, -1.2874, -1.2335],
         [ 0.1629, -0.2907,  2.4620,  ...,  3.5189,  1.1837, -0.7146],
         [-1.3705,  2.4816, -0.0903,  ...,  2.9574,  0.1628, -1.5595]]],
       device='cuda:0', grad_fn=<NativeLayerNormBackward0>)

In [12]:
# Predictor features: what V-JEPA 2 predicts for masked regions — the
# signal that makes it useful as a world model.
# V-JEPA 2 predictor outputs
predictor_outputs = outputs.predictor_output.last_hidden_state

In [13]:
predictor_outputs

tensor([[[-0.0871, -0.0537, -0.0395,  ...,  0.8404,  0.1004, -0.3077],
         [-0.0852, -0.0927, -0.0121,  ...,  0.8539,  0.0474, -0.2956],
         [-0.0067, -0.1536,  0.1253,  ...,  0.8385,  0.0147, -0.2757],
         ...,
         [-0.1527, -0.3922,  0.0330,  ...,  0.1634, -0.0400, -0.2933],
         [-0.1656, -0.3679,  0.0708,  ...,  0.1625, -0.0653, -0.3156],
         [-0.1930, -0.3604,  0.1020,  ...,  0.2005, -0.0755, -0.3181]]],
       device='cuda:0', grad_fn=<ViewBackward0>)

## 2. Video inference (action recognition)

Now we use a checkpoint with a **classification head** fine-tuned on Something-Something-v2 (`fpc16` = 16 frames per clip) and read off the top-5 predicted actions.

In [14]:
import numpy as np
import torch
from torchcodec.decoders import VideoDecoder

from transformers import AutoModelForVideoClassification, AutoVideoProcessor

In [15]:
model_id = "facebook/vjepa2-vitl-fpc16-256-ssv2"

In [16]:
# AutoModelForVideoClassification loads backbone + trained action head.
# Load model and video preprocessor
model = AutoModelForVideoClassification.from_pretrained(model_id, device_map="auto")
processor = AutoVideoProcessor.from_pretrained(model_id)

Loading weights:   0%|          | 0/652 [00:00<?, ?it/s]

In [17]:
# To load a video, 
video_url = "https://huggingface.co/datasets/nateraw/kinetics-mini/resolve/main/val/bowling/-WH-lxmGJVY_000005_000015.mp4"

In [18]:
# Match the model's expected frame count (config.frames_per_clip),
# subsampling every 8th frame here.
# Sample the number of frames according to the model.
vr = VideoDecoder(video_url)
frame_idx = np.arange(0, model.config.frames_per_clip, 8) # you can define more complex sampling strategy
video = vr.get_frames_at(indices=frame_idx).data  # frames x channels x height x width

In [19]:
# Preprocess and run inference
inputs = processor(video, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits

In [20]:
# Convert logits -> softmax probabilities and map indices to human labels.
print("Top 5 predicted class names:")
top5_indices = logits.topk(5).indices[0]

top5_probs = torch.softmax(logits, dim=-1).topk(5).values[0]

for idx, prob in zip(top5_indices, top5_probs):
    text_label = model.config.id2label[idx.item()]
    print(f" - {text_label}: {prob:.2f}")

Top 5 predicted class names:
 - Stuffing [something] into [something]: 0.27
 - Putting [something] into [something]: 0.19
 - Plugging [something] into [something]: 0.10
 - Closing [something]: 0.09
 - Putting [something] onto [something]: 0.04


## 3. Video classification (explicit class + training loss)

Same task via the concrete `VJEPA2ForVideoClassification` class. This section uses a dummy all-ones video just to show the shapes and how to get a **training loss** by passing `labels=`.

In [1]:
import torch

import numpy as np
from transformers import AutoVideoProcessor, VJEPA2ForVideoClassification

In [2]:
model_id = "facebook/vjepa2-vitl-fpc16-256-ssv2"

In [3]:
device = "cuda"

video_processor = AutoVideoProcessor.from_pretrained(model_id)
model = VJEPA2ForVideoClassification.from_pretrained(model_id).to(device)

Loading weights:   0%|          | 0/652 [00:00<?, ?it/s]

In [4]:
# Placeholder video (all ones) — replace with real decoded frames.
video = np.ones((64, 256, 256, 3))  # 64 frames, 256x256 RGB

inputs = video_processor(video, return_tensors="pt").to(device)

In [5]:
# For inference
with torch.no_grad():
    outputs = model(**inputs)
logits = outputs.logits

predicted_label = logits.argmax(-1).item()
print(model.config.id2label[predicted_label])

Pouring [something] into [something]


In [6]:
# Passing labels makes the model return .loss, ready for backprop.
# For training
labels = torch.ones(1, dtype=torch.long, device=device)
loss = model(**inputs, labels=labels).loss